# **Basic Attention LSTM model:**

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTALL DEPENDENCIES (run once in Google Colab)
# ─────────────────────────────────────────────────────────────────────────────
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy

"""
================================================================================
  Attention-LSTM Multi-Horizon Wave Forecasting Model
  For: KBS Paper — Decision Support System (DSS) Pipeline Phase 6
  Architecture: Bahdanau (Additive) Attention + LSTM
  Author: Generated for Coastal/Marine Engineering Research
  Target: Significant Wave Height (Hs) — 3-hourly resolution
  Horizons: [3, 6, 12, 24] hours
  Platform: Google Colab + PyTorch
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & ENVIRONMENT SETUP
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import time
import warnings
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns

warnings.filterwarnings("ignore")

# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ─── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n{'='*70}")
print(f"  PyTorch version : {torch.__version__}")
print(f"  Device          : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"{'='*70}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 1. PATHS & HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS")
OUTPUT_DIR = Path("/content/drive/MyDrive/KBS_Paper/Outputs/6_LSTM_KBS")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Output directory: {OUTPUT_DIR}")

TARGET_COL   = "target_buoy_hs"
HORIZONS     = [3, 6, 12, 24]           # forecast horizons in hours
TIME_RES     = 3                        # hours per sample
TOP_K_FEAT   = 60                       # SelectKBest k
SEQ_LEN      = 8                        # look-back window (8 × 3h = 24h history)
HIDDEN_SIZE  = 64
NUM_LAYERS   = 1
BATCH_SIZE   = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS   = 50
PATIENCE     = 5                        # early stopping patience
VAL_SPLIT    = 0.20
PLOT_HORIZON = 6                        # horizon used for Q1 journal figures

# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("[STEP 2] Loading feature-engineered datasets …")

df_train = pd.read_csv(INPUT_DIR / "X_train_KBS.csv", index_col=0, parse_dates=True)
df_oos   = pd.read_csv(INPUT_DIR / "X_oos_KBS.csv",   index_col=0, parse_dates=True)

print(f"  Train shape : {df_train.shape}")
print(f"  OOS   shape : {df_oos.shape}")

if TARGET_COL not in df_train.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. "
                     f"Available: {list(df_train.columns[:10])} …")

# Separate target before feature selection
y_train_raw = df_train[TARGET_COL].values.reshape(-1, 1)
y_oos_raw   = df_oos[TARGET_COL].values.reshape(-1, 1)

X_train_raw = df_train.drop(columns=[TARGET_COL])
X_oos_raw   = df_oos.drop(columns=[TARGET_COL])

# Store time indices for output alignment
time_train = df_train.index
time_oos   = df_oos.index

# ─────────────────────────────────────────────────────────────────────────────
# 3. FEATURE SELECTION — SelectKBest (f_regression)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[STEP 3] SelectKBest → keeping top {TOP_K_FEAT} features …")

selector = SelectKBest(score_func=f_regression, k=TOP_K_FEAT)
selector.fit(X_train_raw.fillna(0), y_train_raw.ravel())

selected_mask = selector.get_support()
selected_cols = X_train_raw.columns[selected_mask].tolist()
print(f"  Selected features : {selected_cols[:5]} … ({len(selected_cols)} total)")

X_train_sel = X_train_raw[selected_cols].fillna(0).values
X_oos_sel   = X_oos_raw[selected_cols].fillna(0).values

# ─────────────────────────────────────────────────────────────────────────────
# 4. SCALING — MinMaxScaler fitted on TRAIN only
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 4] MinMaxScaler normalization [0, 1] …")

feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_scaled = feat_scaler.fit_transform(X_train_sel)
X_oos_scaled   = feat_scaler.transform(X_oos_sel)

y_train_scaled = target_scaler.fit_transform(y_train_raw)
y_oos_scaled   = target_scaler.transform(y_oos_raw)

print(f"  X_train scaled: {X_train_scaled.shape}, X_oos scaled: {X_oos_scaled.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. SEQUENCE GENERATION — sliding window, no leakage
# ─────────────────────────────────────────────────────────────────────────────
def create_sequences(X: np.ndarray, y: np.ndarray,
                     seq_len: int, horizon_steps: int):
    """
    Build sliding-window sequences for LSTM input.

    Parameters
    ----------
    X             : (T, F) feature matrix (scaled)
    y             : (T, 1) target vector  (scaled)
    seq_len       : look-back window length
    horizon_steps : forecast lead (in time steps)

    Returns
    -------
    X_seq  : (N, seq_len, F)  — input sequences
    y_seq  : (N,)             — target values
    idx    : indices of the *predicted* time step in the original array
    """
    Xs, ys, idxs = [], [], []
    end = len(X) - horizon_steps
    for i in range(seq_len, end + 1):
        Xs.append(X[i - seq_len : i, :])
        ys.append(y[i + horizon_steps - 1, 0])
        idxs.append(i + horizon_steps - 1)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(idxs)


# ─────────────────────────────────────────────────────────────────────────────
# 6. ATTENTION-LSTM ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """
    Bahdanau (Additive) Attention Mechanism.

    Computes alignment scores e_t = v^T * tanh(W_h * h_t + b)
    then softmax over the sequence to get a context vector.

    Reference: Bahdanau et al. (2015) https://arxiv.org/abs/1409.0473
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W_h  = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v    = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor):
        """
        Parameters
        ----------
        hidden_states : (batch, seq_len, hidden_size)

        Returns
        -------
        context       : (batch, hidden_size)
        attn_weights  : (batch, seq_len)
        """
        # energy: (batch, seq_len, hidden_size) → (batch, seq_len, 1)
        energy  = torch.tanh(self.W_h(hidden_states))
        scores  = self.v(energy).squeeze(-1)             # (batch, seq_len)
        weights = torch.softmax(scores, dim=-1)          # (batch, seq_len)
        # context: weighted sum of hidden states
        context = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)  # (batch, hidden)
        return context, weights


class AttentionLSTM(nn.Module):
    """
    LSTM + Bahdanau Attention for single-output regression.

    Architecture:
        Input → LSTM (all hidden states) → Attention → FC → Hs prediction
    """
    def __init__(self, input_size: int, hidden_size: int,
                 num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.attention = BahdanauAttention(hidden_size)
        self.fc        = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor):
        """
        Parameters
        ----------
        x : (batch, seq_len, input_size)

        Returns
        -------
        out          : (batch, 1)
        attn_weights : (batch, seq_len)
        """
        lstm_out, _ = self.lstm(x)                   # (batch, seq_len, hidden)
        context, attn_weights = self.attention(lstm_out)
        out = self.fc(context)                        # (batch, 1)
        return out, attn_weights


# ─────────────────────────────────────────────────────────────────────────────
# 7. TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    """Stops training when validation loss stops improving."""
    def __init__(self, patience: int = 5, delta: float = 1e-6):
        self.patience   = patience
        self.delta      = delta
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_state = None
        self.stopped    = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss - self.delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped

    def restore_best(self, model: nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute RMSE, MAE, R², MAPE in physical units."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # MAPE — guard against near-zero Hs values (< 0.05 m)
    mask = np.abs(y_true) > 0.05
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}


# ─────────────────────────────────────────────────────────────────────────────
# 8. MULTI-HORIZON TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
metrics_records   = []
train_pred_frames = []
oos_pred_frames   = []

# Storage for the +6h model/data (for XAI plots)
model_6h         = None
X_oos_tensor_6h  = None
attn_weights_6h  = None
oos_frame_6h     = None

for horizon_h in HORIZONS:
    horizon_steps = horizon_h // TIME_RES  # number of time steps
    print(f"\n{'='*70}")
    print(f"  HORIZON: +{horizon_h}h  ({horizon_steps} steps @ {TIME_RES}h resolution)")
    print(f"{'='*70}")

    # ── Build sequences ────────────────────────────────────────────────────
    X_tr_seq, y_tr_seq, idx_tr = create_sequences(
        X_train_scaled, y_train_scaled, SEQ_LEN, horizon_steps)
    X_oos_seq, y_oos_seq, idx_oos = create_sequences(
        X_oos_scaled, y_oos_scaled, SEQ_LEN, horizon_steps)

    print(f"  Train sequences : {X_tr_seq.shape}  |  OOS sequences : {X_oos_seq.shape}")

    # ── Train / Validation split (20% from train) ──────────────────────────
    tr_idx, val_idx = train_test_split(
        np.arange(len(X_tr_seq)), test_size=VAL_SPLIT,
        random_state=SEED, shuffle=True)

    X_tr  = torch.tensor(X_tr_seq[tr_idx],  device=DEVICE)
    y_tr  = torch.tensor(y_tr_seq[tr_idx],  device=DEVICE).unsqueeze(1)
    X_val = torch.tensor(X_tr_seq[val_idx], device=DEVICE)
    y_val = torch.tensor(y_tr_seq[val_idx], device=DEVICE).unsqueeze(1)
    X_oos_t = torch.tensor(X_oos_seq,       device=DEVICE)
    y_oos_t = torch.tensor(y_oos_seq,       device=DEVICE).unsqueeze(1)

    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(
        TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    # ── Instantiate model ──────────────────────────────────────────────────
    model = AttentionLSTM(
        input_size  = X_tr_seq.shape[2],
        hidden_size = HIDDEN_SIZE,
        num_layers  = NUM_LAYERS
    ).to(DEVICE)

    optimizer     = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion     = nn.MSELoss()
    early_stopper = EarlyStopping(patience=PATIENCE)

    print(f"  Model parameters: "
          f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # ── Epoch loop ─────────────────────────────────────────────────────────
    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        # — Train —
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred, _ = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(tr_idx)

        # — Validate —
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                pred, _ = model(xb)
                val_loss += criterion(pred, yb).item() * len(xb)
        val_loss /= len(val_idx)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch [{epoch:3d}/{MAX_EPOCHS}]  "
                  f"Train Loss: {train_loss:.6f}  |  Val Loss: {val_loss:.6f}")

        if early_stopper.step(val_loss, model):
            print(f"  [EarlyStopping] Triggered at epoch {epoch} "
                  f"(best val loss: {early_stopper.best_loss:.6f})")
            break

    early_stopper.restore_best(model)
    elapsed = time.time() - t0
    print(f"  Training complete in {elapsed:.1f}s")

    # ── Save model weights ─────────────────────────────────────────────────
    weight_path = OUTPUT_DIR / f"lstm_attention_model_{horizon_h}h.pt"
    torch.save(model.state_dict(), weight_path)
    print(f"  Model saved → {weight_path.name}")

    if horizon_h == PLOT_HORIZON:
        model_6h        = model
        X_oos_tensor_6h = X_oos_t

    # ── Generate predictions ───────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        # Full train set (tr + val for reporting)
        X_all_tr   = torch.tensor(X_tr_seq, device=DEVICE)
        pred_tr_sc, _ = model(X_all_tr)
        pred_tr_sc = pred_tr_sc.cpu().numpy()

        pred_oos_sc, attn_w = model(X_oos_t)
        pred_oos_sc = pred_oos_sc.cpu().numpy()
        attn_np     = attn_w.cpu().numpy()   # (N_oos, seq_len)

    if horizon_h == PLOT_HORIZON:
        attn_weights_6h = attn_np

    # Inverse-transform to physical units (metres)
    pred_tr_phys  = target_scaler.inverse_transform(pred_tr_sc).ravel()
    actual_tr_phys = target_scaler.inverse_transform(y_tr_seq.reshape(-1, 1)).ravel()

    pred_oos_phys  = target_scaler.inverse_transform(pred_oos_sc).ravel()
    actual_oos_phys = target_scaler.inverse_transform(y_oos_seq.reshape(-1, 1)).ravel()

    # Metrics
    m_tr  = compute_metrics(actual_tr_phys,  pred_tr_phys)
    m_oos = compute_metrics(actual_oos_phys, pred_oos_phys)

    print(f"\n  ── TRAIN  RMSE={m_tr['RMSE']:.4f}m  MAE={m_tr['MAE']:.4f}m  "
          f"R²={m_tr['R2']:.4f}  MAPE={m_tr['MAPE']:.2f}%")
    print(f"  ── OOS    RMSE={m_oos['RMSE']:.4f}m  MAE={m_oos['MAE']:.4f}m  "
          f"R²={m_oos['R2']:.4f}  MAPE={m_oos['MAPE']:.2f}%")

    # Record metrics
    for split_name, m in [("Train", m_tr), ("OOS", m_oos)]:
        metrics_records.append({
            "horizon_hours": horizon_h,
            "split"        : split_name,
            "RMSE"         : round(m["RMSE"], 5),
            "MAE"          : round(m["MAE"],  5),
            "R2"           : round(m["R2"],   5),
            "MAPE"         : round(m["MAPE"], 3),
        })

    # ── Prediction DataFrames ──────────────────────────────────────────────
    # Align time index: idx_tr contains original row positions of predictions
    time_tr_pred  = time_train[idx_tr]
    time_oos_pred = time_oos[idx_oos]

    df_tr_pred = pd.DataFrame({
        "time"          : time_tr_pred,
        "split"         : "Train",
        "horizon_hours" : horizon_h,
        "actual_hs"     : actual_tr_phys,
        "predicted_hs"  : pred_tr_phys,
    })
    df_oos_pred = pd.DataFrame({
        "time"          : time_oos_pred,
        "split"         : "OOS",
        "horizon_hours" : horizon_h,
        "actual_hs"     : actual_oos_phys,
        "predicted_hs"  : pred_oos_phys,
    })

    train_pred_frames.append(df_tr_pred)
    oos_pred_frames.append(df_oos_pred)

    if horizon_h == PLOT_HORIZON:
        oos_frame_6h = df_oos_pred.copy()

# ─────────────────────────────────────────────────────────────────────────────
# 9. EXPORT METRICS & PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("  EXPORTING METRICS & PREDICTION FILES")
print(f"{'='*70}")

df_metrics = pd.DataFrame(metrics_records)
df_metrics.to_csv(OUTPUT_DIR / "lstm_metrics_summary.csv", index=False)
print(f"  Saved: lstm_metrics_summary.csv")
print(df_metrics.to_string(index=False))

df_train_preds = pd.concat(train_pred_frames, ignore_index=True)
df_oos_preds   = pd.concat(oos_pred_frames,   ignore_index=True)

df_train_preds.to_csv(OUTPUT_DIR / "lstm_train_predictions.csv", index=False)
df_oos_preds.to_csv(  OUTPUT_DIR / "lstm_oos_predictions.csv",   index=False)
print(f"\n  Saved: lstm_train_predictions.csv ({len(df_train_preds)} rows)")
print(f"  Saved: lstm_oos_predictions.csv   ({len(df_oos_preds)} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# 10. Q1 JOURNAL VISUALIZATION — +6h OOS Set
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  GENERATING Q1 JOURNAL FIGURES (Horizon: +{PLOT_HORIZON}h, OOS set)")
print(f"{'='*70}")

# Global plot style — Seaborn "ticks" as specified
sns.set_style("ticks")
sns.set_context("paper", font_scale=1.4)
PALETTE = {"actual": "#1A4A7A", "predicted": "#E84B3A", "attn": "#2C7BB6"}
DPI = 600

def save_fig(fig, stem: str):
    """Save figure as both PNG and TIFF at 600 DPI."""
    for ext in ["png", "tiff"]:
        fpath = OUTPUT_DIR / f"{stem}.{ext}"
        fig.savefig(fpath, dpi=DPI, bbox_inches="tight", format=ext)
        print(f"  Saved: {fpath.name}")


# ── Figure 1: Density Scatter (Actual vs Predicted) ──────────────────────────
try:
    print("\n[FIG 1] Density Scatter Plot …")
    act_6  = oos_frame_6h["actual_hs"].values
    pred_6 = oos_frame_6h["predicted_hs"].values

    fig1, ax = plt.subplots(figsize=(6, 6))

    # 2D histogram for density colouring
    h, xedge, yedge = np.histogram2d(act_6, pred_6, bins=60)
    h_norm = h / h.max()
    xc = 0.5 * (xedge[:-1] + xedge[1:])
    yc = 0.5 * (yedge[:-1] + yedge[1:])
    xx, yy = np.meshgrid(xc, yc)
    ax.pcolormesh(xx, yy, h_norm.T, cmap="Blues", alpha=0.85)

    # Scatter with density colour
    from scipy.stats import gaussian_kde
    xy   = np.vstack([act_6, pred_6])
    z    = gaussian_kde(xy)(xy)
    idx_sort = z.argsort()
    sc = ax.scatter(act_6[idx_sort], pred_6[idx_sort],
                    c=z[idx_sort], s=8, cmap="viridis",
                    alpha=0.65, linewidths=0, rasterized=True)
    cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("Kernel Density Estimate", fontsize=11)

    # Identity (1:1) line
    lims = [min(act_6.min(), pred_6.min()) * 0.95,
            max(act_6.max(), pred_6.max()) * 1.05]
    ax.plot(lims, lims, "k--", lw=1.5, label="1:1 line", zorder=5)

    # Best-fit line
    m, b = np.polyfit(act_6, pred_6, 1)
    xfit = np.linspace(lims[0], lims[1], 200)
    ax.plot(xfit, m * xfit + b, color="#E84B3A", lw=1.5,
            linestyle="-", label=f"Fit: y={m:.3f}x+{b:.3f}", zorder=6)

    r2_val  = r2_score(act_6, pred_6)
    rmse_val = np.sqrt(mean_squared_error(act_6, pred_6))
    textstr = f"$R^2$ = {r2_val:.4f}\nRMSE = {rmse_val:.4f} m"
    ax.text(0.05, 0.93, textstr, transform=ax.transAxes,
            fontsize=11, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="grey", alpha=0.85))

    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Observed $H_s$ (m)", fontsize=13)
    ax.set_ylabel("Predicted $H_s$ (m)", fontsize=13)
    ax.set_title(f"Attention-LSTM: Observed vs Predicted $H_s$ (+{PLOT_HORIZON}h, OOS)",
                 fontsize=12)
    ax.legend(fontsize=10, loc="lower right")
    sns.despine(fig=fig1, offset=8)
    save_fig(fig1, "fig1_density_scatter_6h")
    plt.close(fig1)

except Exception as e:
    print(f"  [WARNING] Figure 1 failed: {e}")


# ── Figure 2: Time-Series Overlay — Storm Peak Window (14 days) ──────────────
try:
    print("\n[FIG 2] Storm Peak Time-Series Window …")
    oos_frame_6h_sorted = oos_frame_6h.sort_values("time")
    peak_idx = oos_frame_6h_sorted["actual_hs"].idxmax()
    peak_time = pd.Timestamp(oos_frame_6h_sorted.loc[peak_idx, "time"])
    print(f"  Storm peak: {peak_time}  Hs = {oos_frame_6h_sorted.loc[peak_idx, 'actual_hs']:.3f} m")

    window_start = peak_time - pd.Timedelta(days=7)
    window_end   = peak_time + pd.Timedelta(days=7)
    mask_win     = ((oos_frame_6h_sorted["time"] >= window_start) &
                    (oos_frame_6h_sorted["time"] <= window_end))
    df_win = oos_frame_6h_sorted.loc[mask_win]

    fig2, ax = plt.subplots(figsize=(12, 4.5))
    ax.fill_between(df_win["time"], df_win["actual_hs"],
                    alpha=0.18, color=PALETTE["actual"])
    ax.plot(df_win["time"], df_win["actual_hs"],
            color=PALETTE["actual"], lw=1.8, label="Observed $H_s$")
    ax.plot(df_win["time"], df_win["predicted_hs"],
            color=PALETTE["predicted"], lw=1.8, linestyle="--",
            label=f"Predicted $H_s$ (+{PLOT_HORIZON}h)")
    ax.axvline(peak_time, color="grey", ls=":", lw=1.2, label="Storm Peak")
    ax.set_xlabel("Date", fontsize=13)
    ax.set_ylabel("$H_s$ (m)", fontsize=13)
    ax.set_title(f"Attention-LSTM: Storm Peak 14-Day Window (+{PLOT_HORIZON}h, OOS)",
                 fontsize=12)
    ax.legend(fontsize=10, loc="upper left")
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %d"))
    plt.xticks(rotation=25)
    sns.despine(fig=fig2, offset=8)
    fig2.tight_layout()
    save_fig(fig2, "fig2_storm_timeseries_6h")
    plt.close(fig2)

except Exception as e:
    print(f"  [WARNING] Figure 2 failed: {e}")


# ── Figure 3: Attention Weight Heatmap / Bar Chart (XAI) ─────────────────────
try:
    print("\n[FIG 3] Attention Weight XAI Visualization …")

    # Identify the storm-peak sequence in the OOS set
    oos_frame_6h_sorted = oos_frame_6h.sort_values("time").reset_index(drop=True)
    peak_pos = int(oos_frame_6h_sorted["actual_hs"].idxmax())
    # attn_weights_6h shape: (N_oos, seq_len)
    # peak_pos corresponds to the same index in attn_weights_6h
    storm_attn = attn_weights_6h[peak_pos]  # (seq_len,)

    # Hours back: seq_len steps of 3h each, e.g. [-24, -21, …, -3]
    hours_back = [-SEQ_LEN * TIME_RES + i * TIME_RES for i in range(SEQ_LEN)]
    labels     = [f"{h}h" for h in hours_back]

    fig3, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                              gridspec_kw={"width_ratios": [1.5, 1]})

    # ─ Bar chart (left panel) ─────────────────────────────────────────
    ax_bar = axes[0]
    cmap_bar = plt.cm.get_cmap("YlOrRd")
    norm_bar = Normalize(vmin=storm_attn.min(), vmax=storm_attn.max())
    bar_colors = cmap_bar(norm_bar(storm_attn))
    bars = ax_bar.bar(range(SEQ_LEN), storm_attn,
                      color=bar_colors, edgecolor="grey", linewidth=0.5)
    sm = ScalarMappable(cmap=cmap_bar, norm=norm_bar)
    sm.set_array([])
    cb3 = plt.colorbar(sm, ax=ax_bar, fraction=0.04, pad=0.02)
    cb3.set_label("Attention Weight", fontsize=11)
    ax_bar.set_xticks(range(SEQ_LEN))
    ax_bar.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
    ax_bar.set_xlabel("Lag (hours before prediction time)", fontsize=12)
    ax_bar.set_ylabel("Attention Weight", fontsize=12)
    peak_time_str = pd.Timestamp(oos_frame_6h_sorted.iloc[peak_pos]["time"]).strftime("%Y-%m-%d %H:%M")
    ax_bar.set_title(f"Attention Weights at Storm Peak\n({peak_time_str}, +{PLOT_HORIZON}h)",
                     fontsize=11)
    ax_bar.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))
    sns.despine(ax=ax_bar, offset=8)

    # ─ Heatmap (right panel) ─────────────────────────────────────────
    ax_hm = axes[1]
    hm_data = storm_attn.reshape(1, -1)
    im = ax_hm.imshow(hm_data, cmap="YlOrRd", aspect="auto",
                      vmin=storm_attn.min(), vmax=storm_attn.max())
    plt.colorbar(im, ax=ax_hm, fraction=0.12, pad=0.04, label="Weight")
    ax_hm.set_xticks(range(SEQ_LEN))
    ax_hm.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax_hm.set_yticks([])
    ax_hm.set_xlabel("Lag (hours before prediction time)", fontsize=12)
    ax_hm.set_title("Attention Heatmap", fontsize=11)
    for j in range(SEQ_LEN):
        ax_hm.text(j, 0, f"{storm_attn[j]:.3f}",
                   ha="center", va="center", fontsize=8.5,
                   color="black" if storm_attn[j] < storm_attn.max() * 0.7 else "white")

    plt.suptitle(f"XAI — Bahdanau Attention: Which Past Time-Steps Does the Model Focus On? "
                 f"(Horizon: +{PLOT_HORIZON}h)", fontsize=11, y=1.01)
    fig3.tight_layout()
    save_fig(fig3, "fig3_attention_weights_6h")
    plt.close(fig3)

except Exception as e:
    print(f"  [WARNING] Figure 3 failed: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# 11. FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("  PIPELINE COMPLETE — OUTPUT SUMMARY")
print(f"{'='*70}")
print(f"\n  Output directory: {OUTPUT_DIR}\n")

artifacts = [
    "lstm_attention_model_3h.pt",
    "lstm_attention_model_6h.pt",
    "lstm_attention_model_12h.pt",
    "lstm_attention_model_24h.pt",
    "lstm_metrics_summary.csv",
    "lstm_train_predictions.csv",
    "lstm_oos_predictions.csv",
    "fig1_density_scatter_6h.png",
    "fig1_density_scatter_6h.tiff",
    "fig2_storm_timeseries_6h.png",
    "fig2_storm_timeseries_6h.tiff",
    "fig3_attention_weights_6h.png",
    "fig3_attention_weights_6h.tiff",
]
for a in artifacts:
    status = "✓" if (OUTPUT_DIR / a).exists() else "✗ (check for errors above)"
    print(f"  [{status}] {a}")

print(f"\n{'='*70}")
print("  FINAL METRICS TABLE")
print(f"{'='*70}")
print(df_metrics.to_string(index=False))
print(f"\n{'='*70}\n")


  PyTorch version : 2.10.0+cu128
  Device          : cuda
  GPU             : Tesla T4
  VRAM            : 15.64 GB

[INFO] Output directory: /content/drive/MyDrive/KBS_Paper/Outputs/6_LSTM_KBS
[STEP 2] Loading feature-engineered datasets …
  Train shape : (12477, 722)
  OOS   shape : (5605, 722)

[STEP 3] SelectKBest → keeping top 60 features …
  Selected features : ['offshore_26_hs', 'offshore_34_hs', 'offshore_46_hs', 'offshore_56_hs', 'offshore_79_hs'] … (60 total)

[STEP 4] MinMaxScaler normalization [0, 1] …
  X_train scaled: (12477, 60), X_oos scaled: (5605, 60)

  HORIZON: +3h  (1 steps @ 3h resolution)
  Train sequences : (12469, 8, 60)  |  OOS sequences : (5597, 8, 60)
  Model parameters: 36,545
  Epoch [  1/50]  Train Loss: 0.006330  |  Val Loss: 0.003813
  Epoch [  5/50]  Train Loss: 0.003482  |  Val Loss: 0.003058
  Epoch [ 10/50]  Train Loss: 0.003199  |  Val Loss: 0.002688
  Epoch [ 15/50]  Train Loss: 0.003064  |  Val Loss: 0.002635
  Epoch [ 20/50]  Train Loss: 0.0030

# **Optimized Version for the Attention LSTM Model:**

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTALL DEPENDENCIES (uncomment and run once in Google Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy pandas scikit-learn matplotlib seaborn scipy

"""
================================================================================
  Attention-LSTM Multi-Horizon Wave Forecasting — OPTIMIZED
  Target Journal  : Knowledge-Based Systems (Q1, Elsevier)
  Architecture    : Bahdanau Attention + LSTM
  Optimizations   : AMP · Gradient Clipping · ReduceLROnPlateau
                    pin_memory DataLoaders · Early Stopping
  Author          : Generated for Coastal/Marine Engineering Research (KBS Paper)
  Platform        : Google Colab T4 GPU · PyTorch
  Horizons        : [3, 6, 12, 24] hours  |  Time resolution: 3-hourly
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time, warnings, random
from pathlib import Path

import numpy  as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp        import autocast, GradScaler          # AMP

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing     import MinMaxScaler
from sklearn.metrics           import (mean_squared_error,
                                       mean_absolute_error, r2_score)
from sklearn.model_selection   import train_test_split

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.ticker    as ticker
import matplotlib.dates     as mdates
from   matplotlib.colors    import Normalize
from   matplotlib.cm        import ScalarMappable
import seaborn              as sns
from   scipy.stats          import gaussian_kde

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  REPRODUCIBILITY & DEVICE
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED);  np.random.seed(SEED);  torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False          # deterministic > speed

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP    = DEVICE.type == "cuda"                      # AMP only on CUDA
PIN_MEM    = DEVICE.type == "cuda"
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

print(f"\n{'='*70}")
print(f"  PyTorch  : {torch.__version__}")
print(f"  Device   : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {props.total_memory/1e9:.2f} GB")
print(f"  AMP      : {'Enabled' if USE_AMP else 'Disabled (CPU fallback)'}")
print(f"{'='*70}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  PATHS & HYPER-PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS")
OUTPUT_DIR = Path("/content/drive/MyDrive/KBS_Paper/Outputs/6_LSTM_KBS_Optimized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Output directory : {OUTPUT_DIR}\n")

TARGET_COL    = "target_buoy_hs"
HORIZONS      = [3, 6, 12, 24]
TIME_RES      = 3                   # hours per sample
TOP_K_FEAT    = 60
SEQ_LEN       = 8                   # 8 × 3 h = 24 h history
HIDDEN_SIZE   = 64
NUM_LAYERS    = 1
BATCH_SIZE    = 64
LR_INIT       = 1e-3
LR_FACTOR     = 0.5                 # ReduceLROnPlateau factor
LR_PATIENCE   = 3                   # epochs before LR step-down
MAX_EPOCHS    = 50
ES_PATIENCE   = 6                   # early-stopping patience
VAL_SPLIT     = 0.20
GRAD_CLIP     = 1.0
PLOT_HORIZON  = 6

# ─────────────────────────────────────────────────────────────────────────────
# 3.  LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("[STEP 3] Loading datasets …")
df_train = pd.read_csv(INPUT_DIR / "X_train_KBS.csv", index_col=0, parse_dates=True)
df_oos   = pd.read_csv(INPUT_DIR / "X_oos_KBS.csv",   index_col=0, parse_dates=True)
print(f"  Train : {df_train.shape}   OOS : {df_oos.shape}")

if TARGET_COL not in df_train.columns:
    raise ValueError(f"'{TARGET_COL}' not found. First cols: {df_train.columns[:8].tolist()}")

# Drop all target-family columns from features to prevent leakage
other_targets = [c for c in df_train.columns if c.startswith("target_")]
print(f"  Dropping {len(other_targets)} target columns from feature matrix: {other_targets}")

y_train_raw = df_train[TARGET_COL].values.reshape(-1, 1)
y_oos_raw   = df_oos[TARGET_COL].values.reshape(-1, 1)
X_train_raw = df_train.drop(columns=other_targets)
X_oos_raw   = df_oos.drop(columns=other_targets)

time_train  = df_train.index
time_oos    = df_oos.index
print(f"  Feature cols after drop : {X_train_raw.shape[1]}")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  FEATURE SELECTION — SelectKBest (f_regression, fit on TRAIN only)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[STEP 4] SelectKBest → top {TOP_K_FEAT} features …")
selector = SelectKBest(f_regression, k=TOP_K_FEAT)
selector.fit(X_train_raw.fillna(0), y_train_raw.ravel())
selected_cols = X_train_raw.columns[selector.get_support()].tolist()
print(f"  Top-5 selected : {selected_cols[:5]}")

X_train_sel = X_train_raw[selected_cols].fillna(0).values
X_oos_sel   = X_oos_raw[selected_cols].fillna(0).values

# ─────────────────────────────────────────────────────────────────────────────
# 5.  SCALING — MinMaxScaler [0, 1] (fit on TRAIN only)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 5] MinMaxScaler normalisation …")
feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_sc = feat_scaler.fit_transform(X_train_sel)
X_oos_sc   = feat_scaler.transform(X_oos_sel)
y_train_sc = target_scaler.fit_transform(y_train_raw)
y_oos_sc   = target_scaler.transform(y_oos_raw)
print(f"  X_train : {X_train_sc.shape}   X_oos : {X_oos_sc.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  SEQUENCE BUILDER — no leakage across splits
# ─────────────────────────────────────────────────────────────────────────────
def create_sequences(X: np.ndarray, y: np.ndarray,
                     seq_len: int, horizon_steps: int):
    """
    Sliding-window sequence generator.

    Returns
    -------
    X_seq  : (N, seq_len, F)   float32
    y_seq  : (N,)              float32
    idxs   : (N,)   original row index of the *predicted* timestep
    """
    Xs, ys, idxs = [], [], []
    end = len(X) - horizon_steps
    for i in range(seq_len, end + 1):
        Xs.append(X[i - seq_len : i])
        ys.append(y[i + horizon_steps - 1, 0])
        idxs.append(i + horizon_steps - 1)
    return (np.array(Xs,    dtype=np.float32),
            np.array(ys,    dtype=np.float32),
            np.array(idxs,  dtype=np.int64))

# ─────────────────────────────────────────────────────────────────────────────
# 7.  ATTENTION-LSTM ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """
    Additive (Bahdanau) Attention over LSTM hidden states.

    e_t = v^T · tanh(W_h · h_t + b)
    α   = softmax(e)
    c   = Σ α_t · h_t

    Reference: Bahdanau et al. (2015) arXiv:1409.0473
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1,           bias=False)

    def forward(self, H: torch.Tensor):
        """
        H : (B, T, H_dim)
        Returns
            context : (B, H_dim)
            weights : (B, T)        ← interpretable for XAI
        """
        scores  = self.v(torch.tanh(self.W(H))).squeeze(-1)   # (B, T)
        weights = torch.softmax(scores, dim=-1)                # (B, T)
        context = torch.bmm(weights.unsqueeze(1), H).squeeze(1)  # (B, H_dim)
        return context, weights


class AttentionLSTM(nn.Module):
    """LSTM → Bahdanau Attention → Linear → scalar Hs forecast."""
    def __init__(self, input_size: int, hidden_size: int, num_layers: int = 1):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size,
                                 num_layers=num_layers, batch_first=True)
        self.attention = BahdanauAttention(hidden_size)
        self.fc        = nn.Linear(hidden_size, 1)
        self._init_weights()

    def _init_weights(self):
        """Xavier uniform for LSTM + linear layers — faster convergence."""
        for name, p in self.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(p)
            elif "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)

    def forward(self, x: torch.Tensor):
        """
        x : (B, T, F)
        Returns  out:(B,1), attn_weights:(B,T)
        """
        H, _             = self.lstm(x)
        context, weights = self.attention(H)
        return self.fc(context), weights

# ─────────────────────────────────────────────────────────────────────────────
# 8.  TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 6, delta: float = 1e-6):
        self.patience   = patience
        self.delta      = delta
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_state = None
        self.stopped    = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss - self.delta:
            self.best_loss  = val_loss
            self.counter    = 0
            # Store best weights on CPU to free GPU memory
            self.best_state = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped

    def restore_best(self, model: nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    mask = np.abs(y_true) > 0.05          # guard near-zero Hs
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask])
                                 / y_true[mask])) * 100)
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}


def make_loaders(X_tr, y_tr, X_val, y_val):
    """Build pin_memory DataLoaders for fast CPU→GPU transfer."""
    tr_ds  = TensorDataset(torch.tensor(X_tr),  torch.tensor(y_tr).unsqueeze(1))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val).unsqueeze(1))
    kw = dict(pin_memory=PIN_MEM, num_workers=NUM_WORKERS,
              persistent_workers=(NUM_WORKERS > 0))
    return (DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,  **kw),
            DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **kw))

# ─────────────────────────────────────────────────────────────────────────────
# 9.  MULTI-HORIZON TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
metrics_records   = []
train_pred_frames = []
oos_pred_frames   = []
model_6h = attn_weights_6h = oos_frame_6h = None

for horizon_h in HORIZONS:
    horizon_steps = horizon_h // TIME_RES
    print(f"\n{'='*70}")
    print(f"  HORIZON : +{horizon_h:2d}h  ({horizon_steps} steps @ {TIME_RES}h)")
    print(f"{'='*70}")

    # ── Sequences ────────────────────────────────────────────────────────────
    X_tr_seq,  y_tr_seq,  idx_tr  = create_sequences(X_train_sc, y_train_sc,
                                                      SEQ_LEN, horizon_steps)
    X_oos_seq, y_oos_seq, idx_oos = create_sequences(X_oos_sc,   y_oos_sc,
                                                      SEQ_LEN, horizon_steps)
    print(f"  Train seqs : {X_tr_seq.shape}   OOS seqs : {X_oos_seq.shape}")

    # ── Train / Val split ────────────────────────────────────────────────────
    tr_idx, val_idx = train_test_split(np.arange(len(X_tr_seq)),
                                       test_size=VAL_SPLIT,
                                       random_state=SEED, shuffle=True)
    train_loader, val_loader = make_loaders(X_tr_seq[tr_idx],  y_tr_seq[tr_idx],
                                            X_tr_seq[val_idx], y_tr_seq[val_idx])

    # ── Model + Optimiser + Scheduler ────────────────────────────────────────
    model = AttentionLSTM(input_size  = X_tr_seq.shape[2],
                          hidden_size = HIDDEN_SIZE,
                          num_layers  = NUM_LAYERS).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode="min",
                    factor   = LR_FACTOR,
                    patience = LR_PATIENCE)
    criterion = nn.MSELoss()
    scaler    = GradScaler(enabled=USE_AMP)   # AMP gradient scaler
    stopper   = EarlyStopping(patience=ES_PATIENCE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params : {n_params:,}")

    # ── Epoch loop ────────────────────────────────────────────────────────────
    t0       = time.time()
    prev_lr  = LR_INIT

    for epoch in range(1, MAX_EPOCHS + 1):

        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)       # faster than zero_grad()

            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                pred, _ = model(xb)
                loss    = criterion(pred, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)                  # unscale before clipping
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item() * len(xb)

        epoch_loss /= len(tr_idx)

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    pred, _ = model(xb)
                    val_loss += criterion(pred, yb).item() * len(xb)
        val_loss /= len(val_idx)

        # ── LR scheduler step ─────────────────────────────────────────────
        scheduler.step(val_loss)
        cur_lr = optimizer.param_groups[0]["lr"]
        lr_tag = f"  ▼ LR → {cur_lr:.2e}" if cur_lr < prev_lr else ""
        prev_lr = cur_lr

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Ep [{epoch:3d}/{MAX_EPOCHS}]  "
                  f"Train {epoch_loss:.6f}  |  Val {val_loss:.6f}  "
                  f"LR {cur_lr:.2e}{lr_tag}")

        if stopper.step(val_loss, model):
            print(f"  [EarlyStopping] Epoch {epoch}  "
                  f"Best val loss : {stopper.best_loss:.6f}")
            break

    stopper.restore_best(model)
    print(f"  Training done in {time.time()-t0:.1f}s")

    # ── Save weights ──────────────────────────────────────────────────────────
    w_path = OUTPUT_DIR / f"lstm_optimized_{horizon_h}h.pt"
    torch.save(model.state_dict(), w_path)
    print(f"  Weights saved → {w_path.name}")

    # ── Inference (full train + OOS) ──────────────────────────────────────────
    model.eval()
    X_oos_t = torch.tensor(X_oos_seq, device=DEVICE)

    with torch.no_grad():
        # Full train set (tr + val)
        X_all_tr  = torch.tensor(X_tr_seq, device=DEVICE)
        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p_tr_sc, _      = model(X_all_tr)
            p_oos_sc, a_w   = model(X_oos_t)

    p_tr_sc  = p_tr_sc.float().cpu().numpy()
    p_oos_sc = p_oos_sc.float().cpu().numpy()
    attn_np  = a_w.float().cpu().numpy()          # (N_oos, seq_len)

    # Inverse-transform → physical metres
    pred_tr_m   = target_scaler.inverse_transform(p_tr_sc).ravel()
    actual_tr_m = target_scaler.inverse_transform(
                      y_tr_seq.reshape(-1, 1)).ravel()
    pred_oos_m  = target_scaler.inverse_transform(p_oos_sc).ravel()
    actual_oos_m= target_scaler.inverse_transform(
                      y_oos_seq.reshape(-1, 1)).ravel()

    # ── Metrics ───────────────────────────────────────────────────────────────
    m_tr  = compute_metrics(actual_tr_m,  pred_tr_m)
    m_oos = compute_metrics(actual_oos_m, pred_oos_m)

    print(f"\n  ▶ TRAIN  RMSE={m_tr['RMSE']:.4f}m  MAE={m_tr['MAE']:.4f}m  "
          f"R²={m_tr['R2']:.4f}  MAPE={m_tr['MAPE']:.2f}%")
    print(f"  ▶ OOS    RMSE={m_oos['RMSE']:.4f}m  MAE={m_oos['MAE']:.4f}m  "
          f"R²={m_oos['R2']:.4f}  MAPE={m_oos['MAPE']:.2f}%")

    for split, m in [("Train", m_tr), ("OOS", m_oos)]:
        metrics_records.append({"horizon_hours": horizon_h, "split": split,
                                 **{k: round(v, 5) for k, v in m.items()}})

    # ── Prediction DataFrames ─────────────────────────────────────────────────
    df_tr_pred  = pd.DataFrame({
        "time"         : time_train[idx_tr],
        "split"        : "Train",
        "horizon_hours": horizon_h,
        "actual_hs"    : actual_tr_m,
        "predicted_hs" : pred_tr_m,
    })
    df_oos_pred = pd.DataFrame({
        "time"         : time_oos[idx_oos],
        "split"        : "OOS",
        "horizon_hours": horizon_h,
        "actual_hs"    : actual_oos_m,
        "predicted_hs" : pred_oos_m,
    })
    train_pred_frames.append(df_tr_pred)
    oos_pred_frames.append(df_oos_pred)

    if horizon_h == PLOT_HORIZON:
        model_6h        = model
        attn_weights_6h = attn_np
        oos_frame_6h    = df_oos_pred.copy()

# ─────────────────────────────────────────────────────────────────────────────
# 10.  EXPORT METRICS & PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}\n  EXPORTING CSVs\n{'='*70}")

df_metrics = pd.DataFrame(metrics_records)
df_metrics.to_csv(OUTPUT_DIR / "lstm_optimized_metrics_summary.csv", index=False)

df_train_all = pd.concat(train_pred_frames, ignore_index=True)
df_oos_all   = pd.concat(oos_pred_frames,   ignore_index=True)
df_train_all.to_csv(OUTPUT_DIR / "lstm_optimized_train_predictions.csv", index=False)
df_oos_all.to_csv(  OUTPUT_DIR / "lstm_optimized_oos_predictions.csv",   index=False)

print(f"  lstm_optimized_metrics_summary.csv")
print(f"  lstm_optimized_train_predictions.csv  ({len(df_train_all):,} rows)")
print(f"  lstm_optimized_oos_predictions.csv    ({len(df_oos_all):,} rows)")
print(f"\n{df_metrics.to_string(index=False)}")

# ─────────────────────────────────────────────────────────────────────────────
# 11.  Q1 JOURNAL FIGURES — +6h OOS set
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  JOURNAL FIGURES  (Horizon +{PLOT_HORIZON}h, OOS set, 600 DPI)")
print(f"{'='*70}")

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.35)
plt.rcParams.update({
    "font.family"   : "DejaVu Sans",
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

C_OBS  = "#1A4A7A"
C_PRED = "#D62728"
DPI    = 600

def save_fig(fig, stem: str):
    for ext in ("png", "tiff"):
        p = OUTPUT_DIR / f"{stem}.{ext}"
        fig.savefig(p, dpi=DPI, bbox_inches="tight", format=ext)
    print(f"  Saved : {stem}.png / .tiff")

# ── FIG 1 : Density Scatter ───────────────────────────────────────────────────
try:
    print("\n[FIG 1] Density scatter …")
    act  = oos_frame_6h["actual_hs"].values
    pred = oos_frame_6h["predicted_hs"].values

    # KDE colour mapping
    xy = np.vstack([act, pred])
    z  = gaussian_kde(xy)(xy)
    o  = z.argsort()

    fig1, ax = plt.subplots(figsize=(5.5, 5.5))
    sc = ax.scatter(act[o], pred[o], c=z[o], s=6, cmap="viridis",
                    alpha=0.7, linewidths=0, rasterized=True)
    cb = fig1.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("Kernel Density", fontsize=10)

    lims = [min(act.min(), pred.min()) * 0.92,
            max(act.max(), pred.max()) * 1.05]
    ax.plot(lims, lims, "k--", lw=1.4, label="1:1 line", zorder=5)

    m_f, b_f = np.polyfit(act, pred, 1)
    xf = np.linspace(*lims, 200)
    ax.plot(xf, m_f*xf + b_f, color=C_PRED, lw=1.4,
            label=f"Fit  y={m_f:.3f}x+{b_f:.3f}", zorder=6)

    r2v   = r2_score(act, pred)
    rmse_v = np.sqrt(mean_squared_error(act, pred))
    ax.text(0.04, 0.94,
            f"$R^2$ = {r2v:.4f}\nRMSE = {rmse_v:.4f} m",
            transform=ax.transAxes, fontsize=10, va="top",
            bbox=dict(boxstyle="round,pad=0.35", fc="white",
                      ec="grey", alpha=0.85))

    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Observed $H_s$ (m)",  fontsize=12)
    ax.set_ylabel("Predicted $H_s$ (m)", fontsize=12)
    ax.set_title(f"Attention-LSTM: Observed vs Predicted $H_s$\n"
                 f"(+{PLOT_HORIZON}h horizon, OOS set)",  fontsize=11)
    ax.legend(fontsize=9, loc="lower right")
    sns.despine(fig=fig1, offset=6)
    save_fig(fig1, "fig1_optimized_density_scatter_6h")
    plt.close(fig1)

except Exception as e:
    print(f"  [WARNING] Fig 1 failed: {e}")

# ── FIG 2 : Storm-Peak 14-day Time-Series ────────────────────────────────────
try:
    print("\n[FIG 2] Storm-peak 14-day time-series …")
    df6 = oos_frame_6h.sort_values("time").reset_index(drop=True)
    df6["time"] = pd.to_datetime(df6["time"])

    peak_i    = int(df6["actual_hs"].idxmax())
    peak_time = df6.loc[peak_i, "time"]
    print(f"  Storm peak : {peak_time}   Hs = {df6.loc[peak_i,'actual_hs']:.3f} m")

    w0 = peak_time - pd.Timedelta(days=7)
    w1 = peak_time + pd.Timedelta(days=7)
    dw = df6[(df6["time"] >= w0) & (df6["time"] <= w1)]

    fig2, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(dw["time"], dw["actual_hs"],
                    alpha=0.15, color=C_OBS)
    ax.plot(dw["time"], dw["actual_hs"],
            color=C_OBS, lw=1.8, label="Observed $H_s$", zorder=3)
    ax.plot(dw["time"], dw["predicted_hs"],
            color=C_PRED, lw=1.6, ls="--",
            label=f"Predicted $H_s$ (+{PLOT_HORIZON}h)", zorder=4)
    ax.axvline(peak_time, color="grey", ls=":", lw=1.1, label="Storm Peak")

    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("$H_s$ (m)", fontsize=12)
    ax.set_title(f"Attention-LSTM: Storm-Peak 14-Day Window "
                 f"(+{PLOT_HORIZON}h, OOS)", fontsize=11)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.xticks(rotation=25)
    sns.despine(fig=fig2, offset=6)
    fig2.tight_layout()
    save_fig(fig2, "fig2_optimized_storm_timeseries_6h")
    plt.close(fig2)

except Exception as e:
    print(f"  [WARNING] Fig 2 failed: {e}")

# ── FIG 3 : XAI Attention Heatmap at Storm Peak ───────────────────────────────
try:
    print("\n[FIG 3] XAI attention heatmap …")
    df6 = oos_frame_6h.sort_values("time").reset_index(drop=True)
    peak_pos = int(df6["actual_hs"].idxmax())

    storm_attn = attn_weights_6h[peak_pos]   # (seq_len,)
    peak_ts    = pd.Timestamp(df6.loc[peak_pos, "time"])

    hours_back = [-(SEQ_LEN - i) * TIME_RES for i in range(SEQ_LEN)]
    labels     = [f"{h}h" for h in hours_back]

    # Colour norm
    norm_a = Normalize(vmin=storm_attn.min(), vmax=storm_attn.max())
    cmap_a = plt.cm.YlOrRd

    fig3, axes = plt.subplots(1, 2, figsize=(12, 4.2),
                              gridspec_kw={"width_ratios": [1.6, 1]})

    # — Bar chart ──────────────────────────────────────────────────────────
    ax_b = axes[0]
    bar_cols = cmap_a(norm_a(storm_attn))
    ax_b.bar(range(SEQ_LEN), storm_attn, color=bar_cols,
             edgecolor="grey", linewidth=0.5)
    sm_b = ScalarMappable(cmap=cmap_a, norm=norm_a)
    sm_b.set_array([])
    cb_b = fig3.colorbar(sm_b, ax=ax_b, fraction=0.04, pad=0.02)
    cb_b.set_label("Attention Weight", fontsize=10)
    ax_b.set_xticks(range(SEQ_LEN))
    ax_b.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
    ax_b.set_xlabel("Lag (h before forecast time)", fontsize=11)
    ax_b.set_ylabel("Attention Weight",             fontsize=11)
    ax_b.set_title(f"Attention Weights at Storm Peak\n"
                   f"{peak_ts.strftime('%Y-%m-%d %H:%M')} UTC  "
                   f"(+{PLOT_HORIZON}h)", fontsize=10)
    ax_b.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.4f"))
    sns.despine(ax=ax_b, offset=6)

    # — Heatmap ────────────────────────────────────────────────────────────
    ax_h = axes[1]
    im = ax_h.imshow(storm_attn.reshape(1, -1), cmap="YlOrRd",
                     aspect="auto", vmin=storm_attn.min(),
                     vmax=storm_attn.max())
    fig3.colorbar(im, ax=ax_h, fraction=0.15, pad=0.04, label="Weight")
    ax_h.set_xticks(range(SEQ_LEN))
    ax_h.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax_h.set_yticks([])
    ax_h.set_xlabel("Lag (h before forecast time)", fontsize=11)
    ax_h.set_title("Attention Heatmap", fontsize=10)
    thresh = storm_attn.max() * 0.65
    for j in range(SEQ_LEN):
        ax_h.text(j, 0, f"{storm_attn[j]:.4f}",
                  ha="center", va="center", fontsize=8,
                  color="white" if storm_attn[j] > thresh else "black")

    fig3.suptitle(
        f"XAI — Bahdanau Attention: Temporal Focus at Storm Peak "
        f"(Horizon +{PLOT_HORIZON}h, OOS)",
        fontsize=11, y=1.02)
    fig3.tight_layout()
    save_fig(fig3, "fig3_optimized_attention_weights_6h")
    plt.close(fig3)

except Exception as e:
    print(f"  [WARNING] Fig 3 failed: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 12.  FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("  ALL DONE — ARTIFACT CHECKLIST")
print(f"{'='*70}")

artifacts = [
    "lstm_optimized_3h.pt",  "lstm_optimized_6h.pt",
    "lstm_optimized_12h.pt", "lstm_optimized_24h.pt",
    "lstm_optimized_metrics_summary.csv",
    "lstm_optimized_train_predictions.csv",
    "lstm_optimized_oos_predictions.csv",
    "fig1_optimized_density_scatter_6h.png",
    "fig1_optimized_density_scatter_6h.tiff",
    "fig2_optimized_storm_timeseries_6h.png",
    "fig2_optimized_storm_timeseries_6h.tiff",
    "fig3_optimized_attention_weights_6h.png",
    "fig3_optimized_attention_weights_6h.tiff",
]
for a in artifacts:
    tag = "✓" if (OUTPUT_DIR / a).exists() else "✗"
    print(f"  [{tag}] {a}")

print(f"\n{'='*70}")
print("  METRICS SUMMARY")
print(f"{'='*70}")
print(df_metrics.to_string(index=False))
print(f"\n{'='*70}\n")


  PyTorch  : 2.10.0+cu128
  Device   : cuda
  GPU      : Tesla T4
  VRAM     : 15.64 GB
  AMP      : Enabled

[INFO] Output directory : /content/drive/MyDrive/KBS_Paper/Outputs/6_LSTM_KBS_Optimized

[STEP 3] Loading datasets …
  Train : (12477, 722)   OOS : (5605, 722)
  Dropping 103 target columns from feature matrix: ['target_buoy_hs', 'target_buoy_tp', 'target_buoy_mdir', 'target_buoy_windspeed', 'target_buoy_winddir', 'target_buoy_hs_lag_3h', 'target_buoy_hs_lag_6h', 'target_buoy_hs_lag_9h', 'target_buoy_hs_lag_12h', 'target_buoy_hs_lag_15h', 'target_buoy_hs_lag_18h', 'target_buoy_hs_lag_21h', 'target_buoy_hs_lag_24h', 'target_buoy_hs_lag_27h', 'target_buoy_hs_lag_30h', 'target_buoy_tp_lag_3h', 'target_buoy_tp_lag_6h', 'target_buoy_tp_lag_9h', 'target_buoy_tp_lag_12h', 'target_buoy_tp_lag_15h', 'target_buoy_tp_lag_18h', 'target_buoy_tp_lag_21h', 'target_buoy_tp_lag_24h', 'target_buoy_tp_lag_27h', 'target_buoy_tp_lag_30h', 'target_buoy_mdir_lag_3h', 'target_buoy_mdir_lag_6h', 'targ